In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import sys
import os

project_root = os.path.dirname(os.path.abspath(''))
if project_root not in sys.path:
    sys.path.append(project_root)

from FEATURES.features import *
from FEATURES.featuresV2 import *
from PRODUCTION.calculateEVS import *
from PRODUCTION.pipeline import *
from PRODUCTION.teamInfo import teamStarPlayer, projectedStartingFive, mainStartingFive

### Load Model

In [2]:
# Load split NGBoost models (mean, variance, and calibration factor)
pts_mean_model = joblib.load('../MODELS/SAVED_MODELS/NGBOOST_PTS_MEAN_MODEL_PRODUCTION.pkl')
pts_var_model = joblib.load('../MODELS/SAVED_MODELS/NGBOOST_PTS_VAR_MODEL_PRODUCTION.pkl')
calibration_factor = joblib.load('../MODELS/SAVED_MODELS/NGBOOST_PTS_CALIBRATION_FACTOR_PRODUCTION.pkl')
model = (pts_mean_model, pts_var_model, calibration_factor)  
features = joblib.load('../MODELS/SAVED_MODELS/feature_list.pkl')

print(f"Loaded models with calibration factor: {calibration_factor}")

Loaded models with calibration factor: 3.97


### Load Player Data and Bookmaker Data

In [3]:
pd.set_option('display.max_columns', None)
today = datetime.today().strftime('%Y%m%d')  
current_date = datetime.now().strftime('%Y-%m-%d')

s26 = pd.read_csv('../DATA/CSV_FILES/TRAIN_DATA/PTS_TRAIN_26.csv').sort_values(by='GAME_DATE')

usData = pd.read_csv(f'../DATA/CSV_FILES/PROP_DATA/PLAYER_LINES/NBA_US_{today}.csv')
dfsData = pd.read_csv(f'../DATA/CSV_FILES/PROP_DATA/PLAYER_LINES/NBA_DFS_{today}.csv')

dfsData.head()

C:\Users\alexg\AppData\Local\Temp\ipykernel_72192\1321450873.py:5: DtypeWarning: Columns (33) have mixed types. Specify dtype option on import or set low_memory=False.
  s26 = pd.read_csv('../DATA/CSV_FILES/TRAIN_DATA/PTS_TRAIN_26.csv').sort_values(by='GAME_DATE')


,BOOKMAKER,CATEGORY,NAME,OVER/UNDER,LINE,ODDS,COMMENCE_TIME,LAST_UPDATE
0,Underdog,player_points,Jaylen Brown,Over,26.5,-137,2025-11-16,2025-11-16T19:08:02Z
1,Underdog,player_points,Jaylen Brown,Under,26.5,-137,2025-11-16,2025-11-16T19:08:02Z
2,Underdog,player_points,Anfernee Simons,Over,14.5,-137,2025-11-16,2025-11-16T19:08:02Z
3,Underdog,player_points,Anfernee Simons,Under,14.5,-137,2025-11-16,2025-11-16T19:08:02Z
4,Underdog,player_points,James Harden,Over,23.5,-137,2025-11-16,2025-11-16T19:08:02Z


### Update projected starting lineups

In [4]:
from MODELS.scrapStarting import NBADailyLineups

scraper = NBADailyLineups("https://www.rotowire.com/basketball/nba-lineups.php")
scraper.getDict()  # Scrape the lineups
scraper.updateTeamInfo()  # Update teamInfo.py

No data available. Run getDict() first.


### Top EVs for single bets

In [5]:
singlePTSBookies = usData[(usData['CATEGORY'] == 'player_points')]

results = calculateSingleBets(s26, singlePTSBookies, model, features, current_date, 
edge_threshold=0.30, stake=10, variance_inflation=1.1, use_monte_carlo=True, n_simulations=10000, 
max_kelly=0.25)  


singleBets = results.sort_values(by='EV$', ascending=False).reset_index(drop=True)
singleBets = singleBets[['NAME', 'BOOKMAKER','LINE', 'PREDICTION','SIDE','ODDS','RECOMMENDATION', 'EV$', 'EXPECTED ROI', 'KELLY_FRACTION','SIGMA FLAG']].head(15)
singleBets.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/singleBets.csv', index=False)
singleBets.head(5)

Processing single bets with single model...
Pre-computing predictions for 113 unique players...


,NAME,BOOKMAKER,LINE,PREDICTION,SIDE,ODDS,RECOMMENDATION,EV$,EXPECTED ROI,KELLY_FRACTION,SIGMA FLAG
0,Dillon Brooks,Bovada,20.5,22.52,Over,205,0,8.53,85.3,0.416,High
1,Dillon Brooks,Bovada,19.5,22.52,Over,165,0,7.18,71.8,0.435,High
2,Onyeka Okongwu,Bovada,15.5,16.75,Over,200,0,7.08,70.8,0.354,High
3,Naji Marshall,Bovada,14.5,15.10,Over,200,0,6.24,62.4,0.312,High
4,Nikola Vucevic,Bovada,15.5,13.98,Under,185,0,6.19,61.9,0.334,High


## Top EVs for 2 leg bets

### Underdog picks

In [6]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'Underdog') & (dfsData['CATEGORY'] == 'player_points')]

results = calculate2LegBets(s26, dfsPTS, model, features, current_date, edge_threshold=4, stake=10, 
variance_inflation=1.1, use_monte_carlo=True, n_simulations=10000, max_kelly=0.25, max_player_appearances=3)

underdogPairs = results.sort_values(by='EV$', ascending=False).reset_index(drop=True)
underdogPairs = underdogPairs[['NAME 1', 'NAME 2', 'LINE 1', 'LINE 2', 'PREDICTION 1', 'PREDICTION 2', 'MODEL SIDE 1', 'MODEL SIDE 2', 'RECOMMENDATION','EV$', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2']].head(10)
underdogPairs.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/underdogPairs.csv', index=False)
underdogPairs.head()

Pre-computing predictions for 93 players...
Processing 84 players with valid predictions...
Generated 3294 valid 2-leg combinations
Applied player frequency limit (3 max appearances per player)
Selected 125 combinations from 3294 candidates


,NAME 1,NAME 2,LINE 1,LINE 2,PREDICTION 1,PREDICTION 2,MODEL SIDE 1,MODEL SIDE 2,RECOMMENDATION,EV$,KELLY FULL,SIGMA FLAG 1,SIGMA FLAG 2
0,Dillon Brooks,Onyeka Okongwu,17.5,12.5,22.52,16.75,over,over,0,4.56,0.228,High,High
1,Zion Williamson,Dillon Brooks,18.5,17.5,22.27,22.52,over,over,0,4.54,0.227,High,High
2,Dillon Brooks,Lauri Markkanen,17.5,26.5,22.52,31.18,over,over,1,4.46,0.223,High,High
3,Onyeka Okongwu,Lauri Markkanen,12.5,26.5,16.75,31.18,over,over,0,4.29,0.214,High,High
4,Zion Williamson,Onyeka Okongwu,18.5,12.5,22.27,16.75,over,over,0,4.28,0.214,High,High


### Prizepicks picks

In [7]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'PrizePicks') & (dfsData['CATEGORY'] == 'player_points')]

results = calculate2LegBets(s26, dfsPTS, model, features, current_date, edge_threshold=4, stake=10, 
variance_inflation=1.1, use_monte_carlo=True, n_simulations=10000, max_kelly=0.25, max_player_appearances=3)

pairsPrizepicks = results.sort_values(by='EV$', ascending=False).reset_index(drop=True)
pairsPrizepicks = pairsPrizepicks[['NAME 1', 'NAME 2', 'LINE 1', 'LINE 2', 'PREDICTION 1', 'PREDICTION 2', 'MODEL SIDE 1', 'MODEL SIDE 2', 'RECOMMENDATION','EV$', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2']].head(10)
pairsPrizepicks.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/prizepicksPairs.csv', index=False)
pairsPrizepicks.head()

Pre-computing predictions for 102 players...
Processing 92 players with valid predictions...
Generated 3942 valid 2-leg combinations
Applied player frequency limit (3 max appearances per player)
Selected 137 combinations from 3942 candidates


,NAME 1,NAME 2,LINE 1,LINE 2,PREDICTION 1,PREDICTION 2,MODEL SIDE 1,MODEL SIDE 2,RECOMMENDATION,EV$,KELLY FULL,SIGMA FLAG 1,SIGMA FLAG 2
0,Dillon Brooks,Onyeka Okongwu,16.5,12.5,22.52,16.75,over,over,0,5.33,0.266,High,High
1,Nicolas Batum,Dillon Brooks,4.5,16.5,6.26,22.52,over,over,0,5.20,0.260,Med,High
2,Dillon Brooks,Lauri Markkanen,16.5,26.5,22.52,31.18,over,over,1,5.05,0.252,High,High
3,Onyeka Okongwu,Lauri Markkanen,12.5,26.5,16.75,31.18,over,over,0,4.26,0.213,High,High
4,Naji Marshall,Onyeka Okongwu,11.5,12.5,15.10,16.75,over,over,0,4.08,0.204,High,High


## 3 leg parlay

### Underdog picks

In [8]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'Underdog') & (dfsData['CATEGORY'] == 'player_points')]

threeLeg = calculate3LegBets(s26, dfsPTS, model, features, current_date, edge_threshold=4, stake=10, 
variance_inflation=1.1, use_monte_carlo=True, n_simulations=10000, max_kelly=0.25, max_player_appearances=2)

underdogTrios = threeLeg.sort_values(by='EV$', ascending=False).reset_index(drop=True)
underdogTrios = underdogTrios[['NAME 1', 'NAME 2', 'NAME 3', 'LINE 1', 'LINE 2', 'LINE 3', 'PREDICTION 1', 'PREDICTION 2', 'PREDICTION 3', 'MODEL SIDE 1', 'MODEL SIDE 2', 'MODEL SIDE 3', 'RECOMMENDATION','EV$', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2', 'SIGMA FLAG 3']].head(10)
underdogTrios.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/underdogTrios.csv', index=False)
underdogTrios.head()

Pre-computing predictions for 93 players...
Processing 84 players with valid predictions...
Generated 94038 valid 3-leg combinations
Applied player frequency limit (2 max appearances per player)
Selected 55 combinations from 94038 candidates


,NAME 1,NAME 2,NAME 3,LINE 1,LINE 2,LINE 3,PREDICTION 1,PREDICTION 2,PREDICTION 3,MODEL SIDE 1,MODEL SIDE 2,MODEL SIDE 3,RECOMMENDATION,EV$,KELLY FULL,SIGMA FLAG 1,SIGMA FLAG 2,SIGMA FLAG 3
0,Zion Williamson,Dillon Brooks,Onyeka Okongwu,18.5,17.5,12.5,22.27,22.52,16.75,over,over,over,0,9.17,0.183,High,High,High
1,Zion Williamson,Dillon Brooks,Lauri Markkanen,18.5,17.5,26.5,22.27,22.52,31.18,over,over,over,0,8.93,0.179,High,High,High
2,Onyeka Okongwu,Lauri Markkanen,Josh Giddey,12.5,26.5,20.5,16.75,31.18,24.77,over,over,over,0,8.29,0.166,High,High,High
3,Harrison Barnes,Josh Giddey,Kevin Huerter,9.5,20.5,12.5,12.54,24.77,15.89,over,over,over,0,7.13,0.143,High,High,High
4,Naji Marshall,Kevin Huerter,Svi Mykhailiuk,11.5,12.5,8.5,15.10,15.89,11.32,over,over,over,0,6.96,0.139,High,High,High


### Prizepicks picks

In [9]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'PrizePicks') & (dfsData['CATEGORY'] == 'player_points')]

threeLeg = calculate3LegBets(s26, dfsPTS, model, features, current_date, edge_threshold=4, stake=10, 
variance_inflation=1.1, use_monte_carlo=True, n_simulations=10000, max_kelly=0.25, max_player_appearances=2)

triosPrizepicks = threeLeg.sort_values(by='EV$', ascending=False).reset_index(drop=True)
triosPrizepicks = triosPrizepicks[['NAME 1', 'NAME 2', 'NAME 3', 'LINE 1', 'LINE 2', 'LINE 3', 'PREDICTION 1', 'PREDICTION 2', 'PREDICTION 3', 'MODEL SIDE 1', 'MODEL SIDE 2', 'MODEL SIDE 3', 'RECOMMENDATION','EV$', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2', 'SIGMA FLAG 3']].head(10)
triosPrizepicks.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/prizepicksTrios.csv', index=False)
triosPrizepicks.head()

Pre-computing predictions for 102 players...
Processing 92 players with valid predictions...
Generated 123912 valid 3-leg combinations
Applied player frequency limit (2 max appearances per player)
Selected 61 combinations from 123912 candidates


,NAME 1,NAME 2,NAME 3,LINE 1,LINE 2,LINE 3,PREDICTION 1,PREDICTION 2,PREDICTION 3,MODEL SIDE 1,MODEL SIDE 2,MODEL SIDE 3,RECOMMENDATION,EV$,KELLY FULL,SIGMA FLAG 1,SIGMA FLAG 2,SIGMA FLAG 3
0,Nicolas Batum,Dillon Brooks,Onyeka Okongwu,4.5,16.5,12.5,6.26,22.52,16.75,over,over,over,0,9.62,0.192,Med,High,High
1,Harrison Barnes,Dillon Brooks,Onyeka Okongwu,9.5,16.5,12.5,12.54,22.52,16.75,over,over,over,0,9.62,0.192,High,High,High
2,Nicolas Batum,Lauri Markkanen,Josh Giddey,4.5,26.5,20.5,6.26,31.18,24.77,over,over,over,0,8.34,0.167,Med,High,High
3,Naji Marshall,Lauri Markkanen,Josh Giddey,11.5,26.5,20.5,15.10,31.18,24.77,over,over,over,0,7.73,0.155,High,High,High
4,Day'Ron Sharpe,Zion Williamson,Naji Marshall,6.5,19.0,11.5,8.70,22.27,15.10,over,over,over,0,7.10,0.142,Med,High,High
